# 05 — Statistical Analysis

Structured analysis of the DUI-by-state master dataset.

Sections:
1. **Regional differences** — ANOVA + post-hoc tests
2. **What drives fatality rates?** — Multiple regression (OLS)
3. **Policy effectiveness** — IID, felony, enforcement with confounders controlled
4. **State typology** — Clustering into archetypes
5. **Consumption-fatality gap** — Residual analysis (perception vs reality)

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor

%matplotlib inline
sns.set_style('whitegrid')

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)

# Load master table
df = pd.read_parquet(PROJECT / 'export' / 'dui_by_state_v2.parquet')
print(f'Master table: {df.shape[0]} states x {df.shape[1]} columns')
df.head(3)

---
## 1. Regional Differences

Are the regional clusters we see in scatter plots statistically significant?

Tests:
- **Kruskal-Wallis** (non-parametric, no normality assumption) for each key metric
- **Tukey HSD** post-hoc to identify which region pairs differ

In [ ]:
# Metrics to test across regions
metrics = [
    'alcohol_fatality_rate_per_100m_vmt',
    'dui_arrest_rate_per_100k_reporting',
    'ethanol_per_capita_gallons_2022',
    'pct_alcohol_nhtsa_imputed',
]

print('=== Kruskal-Wallis Tests (H0: all regions have the same distribution) ===\n')
for metric in metrics:
    groups = [g[metric].dropna().values for _, g in df.groupby('region')]
    stat, p = stats.kruskal(*groups)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'{metric:<45} H={stat:6.2f}  p={p:.4f}  {sig}')

### How to read this:

- **H** = the test statistic. Higher = more difference between groups.
- **p** = probability that the differences are due to random chance. Lower = more confident the difference is real.
- **Significance markers:** `***` = very strong evidence (p < 0.001), `**` = strong (p < 0.01), `*` = moderate (p < 0.05), `ns` = not significant.

**Interpretation:** If a metric shows `*` or better, the regions genuinely differ on that measure — it's not just random noise. If `ns`, the apparent regional pattern in the scatter plot could just be coincidence.

**Questions this raises:** If regions differ significantly on fatality rate but NOT on consumption, that suggests drinking isn't the explanation for regional differences — it's something else (road design, enforcement, speed limits).

In [ ]:
# Post-hoc: Tukey HSD for the most significant metric
print('=== Tukey HSD: Alcohol Fatality Rate per 100M VMT by Region ===\n')
tukey = pairwise_tukeyhsd(
    df['alcohol_fatality_rate_per_100m_vmt'].dropna(),
    df.loc[df['alcohol_fatality_rate_per_100m_vmt'].notna(), 'region'],
    alpha=0.05,
)
print(tukey.summary())

### How to read Tukey HSD:

Each row compares two regions. Key columns:
- **meandiff** = how much higher/lower group2 is compared to group1
- **p-adj** = adjusted p-value (accounts for multiple comparisons)
- **reject** = True means the difference IS statistically significant at 95% confidence

**Interpretation:** If Northeast vs South shows `reject=True`, those two regions have genuinely different fatality rates — not just random variation. If `reject=False`, we can't confidently say they differ.

**Conclusion you can draw:** "The South has a statistically significantly higher alcohol fatality rate than the Northeast" (or whatever the data shows). This is publishable-strength evidence.

In [ ]:
# Tukey HSD for DUI arrest rate
print('=== Tukey HSD: DUI Arrest Rate per 100k by Region ===\n')
tukey_arrests = pairwise_tukeyhsd(
    df['dui_arrest_rate_per_100k_reporting'].dropna(),
    df.loc[df['dui_arrest_rate_per_100k_reporting'].notna(), 'region'],
    alpha=0.05,
)
print(tukey_arrests.summary())

### Interpretation:

Same logic as above but for DUI arrest rates. If the Northeast clusters tightly on arrests (your observation from the scatter), this test tells you whether that clustering is statistically real.

**Questions this raises:** If region A arrests more but has similar fatality rates to region B, does enforcement actually reduce deaths? Or does it just reflect different policing philosophies?

In [ ]:
# Visual: box plots by region for key metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, metric in zip(axes.flat, metrics):
    sns.boxplot(data=df, x='region', y=metric, ax=ax, palette='Set2')
    ax.set_title(metric.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
fig.suptitle('Key Metrics by Census Region', fontsize=14, fontweight='bold')
fig.tight_layout()
fig

---
## 2. What Drives Alcohol Fatality Rates?

OLS multiple regression: which factors correlate with higher/lower per-VMT fatality rates?

Model: `alcohol_fatality_rate_per_100m_vmt ~ consumption + speed + policy + enforcement`

In [ ]:
# Define model variables
outcome = 'alcohol_fatality_rate_per_100m_vmt'

predictors = [
    'ethanol_per_capita_gallons_2022',   # consumption
    'max_speed_limit_mph',               # infrastructure
    'iid_all_offender',                  # policy: IID mandate
    'checkpoints_permitted',                 # policy: sobriety checkpoints
    'dui_arrest_rate_per_100k_reporting',          # enforcement intensity
    'first_offense_felony',              # deterrence
]

# Prepare data (drop rows with missing values in any predictor)
model_df = df[[outcome] + predictors].dropna().copy()
print(f'Model sample: {len(model_df)} states (dropped {51 - len(model_df)} with missing data)')

# Standardize predictors for comparable coefficients
scaler = StandardScaler()
X_std = pd.DataFrame(
    scaler.fit_transform(model_df[predictors]),
    columns=predictors,
    index=model_df.index,
)
X_std = sm.add_constant(X_std)

# Fit OLS
model = sm.OLS(model_df[outcome], X_std).fit()
print(model.summary())

### How to read the OLS regression output:

**Top section:**
- **R-squared** = what % of the variation in fatality rates is explained by all predictors combined. 0.50 = 50% explained. Higher is better.
- **F-statistic p-value** = is the overall model significant? If p < 0.05, at least one predictor matters.

**Coefficient table:**
- **coef** = the standardized effect size. Since we standardized all variables, these are directly comparable. A coef of 0.40 means that variable has twice the impact of one with coef 0.20.
- **Positive coef** = associated with MORE deaths. Negative = associated with FEWER deaths.
- **P>|t|** = is this individual predictor significant? p < 0.05 = yes.
- **[0.025, 0.975]** = 95% confidence interval for the coefficient.

**Interpretation:** The largest positive coefficient is the strongest risk factor. The largest negative coefficient is the strongest protective factor. If a coefficient is large but p > 0.05, the effect is uncertain (could be noise).

**Conclusions you can draw:** "After controlling for other factors, [variable X] has the strongest association with alcohol fatality rates" — but remember: correlation is not causation. This identifies associations, not causal mechanisms.

In [ ]:
# Coefficient plot (standardized — shows relative importance)
coefs = model.params.drop('const')
ci = model.conf_int().drop('const')
ci.columns = ['lower', 'upper']

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(coefs))
ax.barh(y_pos, coefs.values, color=['#E76F51' if c > 0 else '#2A9D8F' for c in coefs.values], alpha=0.8)
ax.errorbar(coefs.values, y_pos, xerr=[coefs.values - ci['lower'].values, ci['upper'].values - coefs.values],
            fmt='none', color='black', capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels([p.replace('_', ' ').title()[:30] for p in coefs.index])
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Standardized Coefficient (effect on fatality rate)')
ax.set_title('What Drives Alcohol Fatality Rates?\nOLS standardized coefficients', fontweight='bold')
fig.tight_layout()
fig

### How to read the coefficient plot:

- **Coral bars (pointing right)** = associated with HIGHER fatality rates
- **Teal bars (pointing left)** = associated with LOWER fatality rates
- **Error bars** = 95% confidence interval. If the error bar crosses zero (the dashed line), the effect is NOT statistically significant.
- **Bar length** = relative importance. Longer bar = stronger effect.

**This is the money chart** for social media: it directly answers "what actually matters for drunk driving deaths?" in one visual.

In [ ]:
# Check multicollinearity (VIF > 5 = concern, > 10 = serious)
print('=== Variance Inflation Factors ===\n')
X_no_const = X_std.drop('const', axis=1)
for i, col in enumerate(X_no_const.columns):
    vif = variance_inflation_factor(X_no_const.values, i)
    flag = ' ⚠️' if vif > 5 else ''
    print(f'  {col:<40} VIF = {vif:.2f}{flag}')

### How to read VIF:

VIF (Variance Inflation Factor) checks whether your predictors are too correlated with each other — which would make the regression unreliable.

- **VIF = 1** = no correlation with other predictors (ideal)
- **VIF 1–5** = moderate, acceptable
- **VIF > 5** = concerning, results may be unstable
- **VIF > 10** = serious problem, that predictor is almost a duplicate of others

**If you see high VIF:** Consider removing one of the correlated predictors. The regression coefficients for high-VIF variables are unreliable (they bounce around depending on what else is in the model).

---
## 3. Policy Effectiveness (controlling for confounders)

The raw comparison showed IID states and non-IID states have similar fatality rates.
But is that still true after controlling for consumption, speed limits, and region?

In [ ]:
# IID effect, controlling for confounders
formula = ('alcohol_fatality_rate_per_100m_vmt ~ '
           'iid_all_offender + ethanol_per_capita_gallons_2022 + '
           'max_speed_limit_mph + C(region)')

iid_model = smf.ols(formula, data=df).fit()
print('=== IID Effect (controlling for consumption, speed, region) ===\n')
print(iid_model.summary().tables[1])

### How to read this:

This isolates the IID effect while holding other factors constant. Look at the row for `iid_all_offender`:

- **coef** = how much the fatality rate changes when a state has IID-for-all (vs not), holding consumption/speed/region constant
- **Negative coef** = IID states have lower fatality rates (controlling for confounders)
- **P>|t|** = is the effect statistically significant?

**Why this matters:** The raw comparison (Section 4 in the viz notebook) showed IID and non-IID states were basically the same. But that didn't account for the fact that IID states might also be higher-consumption or higher-speed states. This model separates those effects.

**Conclusion:** If the IID coefficient is significant here but wasn't in the raw comparison, that means confounders were masking a real effect. If it's STILL not significant, IID mandates genuinely don't correlate with lower deaths at the state level.

In [ ]:
# Felony effect, controlling for confounders
formula_felony = ('alcohol_fatality_rate_per_100m_vmt ~ '
                  'first_offense_felony + ethanol_per_capita_gallons_2022 + '
                  'max_speed_limit_mph + C(region)')

felony_model = smf.ols(formula_felony, data=df.dropna(subset=['first_offense_felony'])).fit()
print('=== Felony Effect (controlling for consumption, speed, region) ===\n')
print(felony_model.summary().tables[1])

### Interpretation:

Same logic as IID above. The earlier finding was that felony states have HIGHER raw fatality rates — but we suspected that's because felony states tend to be Southern/rural (higher speeds, more driving). This model tests: after removing the region and speed effect, do felony laws still correlate with worse outcomes?

- **Positive coef + significant** = felony states genuinely do worse even after controlling for confounders (policy backfire? or just correlation with unmeasured rurality)
- **Non-significant** = the earlier finding was entirely explained by region/speed — the felony law itself isn't the issue

In [ ]:
# Checkpoint effect
formula_ckpt = ('alcohol_fatality_rate_per_100m_vmt ~ '
                'checkpoints_permitted + ethanol_per_capita_gallons_2022 + '
                'max_speed_limit_mph + C(region)')

ckpt_model = smf.ols(formula_ckpt, data=df).fit()
print('=== Checkpoint Effect (controlling for consumption, speed, region) ===\n')
print(ckpt_model.summary().tables[1])

### Interpretation:

Tests whether states allowing sobriety checkpoints have different fatality rates, after controlling for how much people drink, how fast they drive, and where they are geographically.

- **Negative coef + significant** = checkpoints are associated with fewer deaths (a win for the "enforcement works" narrative)
- **Non-significant** = checkpoints don't make a measurable difference at state level

**Caveat:** Even if significant, this doesn't prove checkpoints CAUSE lower deaths — states that adopt checkpoints might also have other unmeasured pro-safety cultures.

---
## 4. State Typology (Clustering)

Group states into archetypes based on policy + outcomes.
Useful for social media: "What type of DUI state do you live in?"

In [ ]:
# Clustering features
cluster_cols = [
    'alcohol_fatality_rate_per_100m_vmt',
    'dui_arrest_rate_per_100k_reporting',
    'ethanol_per_capita_gallons_2022',
    'iid_all_offender',
    'checkpoints_permitted',
    'max_speed_limit_mph',
]

cluster_df = df[['state_name', 'state_abbr', 'region'] + cluster_cols].dropna().copy()
print(f'Clustering on {len(cluster_df)} states, {len(cluster_cols)} features')

# Standardize
X_cluster = StandardScaler().fit_transform(cluster_df[cluster_cols])

# Elbow plot to pick k
inertias = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(K_range, inertias, 'o-', color='#264653')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Plot for State Typology', fontweight='bold')
fig.tight_layout()
fig

### How to read the elbow plot:

- **Y-axis (Inertia)** = how spread out states are within their assigned cluster. Lower = tighter, more cohesive clusters.
- **The "elbow"** = the point where adding more clusters stops giving meaningful improvement. The line bends from steep to flat.

**How to pick k:** Look for where the curve bends sharply. If it bends at k=3 or k=4, that's your best number of clusters. Don't pick too many — you want groups that are interpretable and nameable for social media.

**If there's no clear elbow:** The data doesn't have strongly distinct groups. You can still pick k=3 or k=4 for narrative purposes, but acknowledge the boundaries are fuzzy.

In [ ]:
# Fit with k=4 (adjust based on elbow)
K = 4
km = KMeans(n_clusters=K, n_init=20, random_state=42)
cluster_df['cluster'] = km.fit_predict(X_cluster)

# Profile each cluster
print(f'=== {K} Cluster Profiles (mean values) ===\n')
profiles = cluster_df.groupby('cluster')[cluster_cols].mean().round(2)
profiles['n_states'] = cluster_df.groupby('cluster').size()
print(profiles.to_string())
print()

# List states per cluster
for c in range(K):
    states = cluster_df[cluster_df['cluster'] == c]['state_abbr'].tolist()
    print(f'Cluster {c}: {", ".join(states)}')

### How to read cluster profiles:

Each row is a cluster. The values are the AVERAGE of each metric for all states in that cluster.

**Name your clusters** by their defining characteristics. Examples:
- High fatality + low arrest + high speed = "Dangerous & Lax"
- Low fatality + high arrest + IID = "Strict & Safe"
- High consumption + low fatality = "Drink but Don't Die" (good infrastructure/enforcement)
- Low everything = "Low-Risk Baseline"

**Social media angle:** "What type of DUI state is yours?" with a map colored by cluster. People love finding their state and seeing what archetype it falls into.

**Questions this raises:** Do the clusters map cleanly onto regions? If so, "state type" might just be a proxy for geography. If clusters cut across regions, that's more interesting — it means policy choices matter independent of location.

In [ ]:
# Scatter: fatality rate vs arrest rate, colored by cluster
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#264653', '#2A9D8F', '#F4A261', '#E76F51']
for c in range(K):
    subset = cluster_df[cluster_df['cluster'] == c]
    ax.scatter(subset['dui_arrest_rate_per_100k_reporting'], subset['alcohol_fatality_rate_per_100m_vmt'],
              c=colors[c], s=60, alpha=0.8, label=f'Cluster {c} (n={len(subset)})', edgecolors='white')
    for _, row in subset.iterrows():
        ax.annotate(row['state_abbr'], (row['dui_arrest_rate_per_100k_reporting'], row['alcohol_fatality_rate_per_100m_vmt']),
                   fontsize=7, ha='left', va='bottom', xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('DUI Arrest Rate per 100k')
ax.set_ylabel('Alcohol Fatality Rate per 100M VMT')
ax.set_title('State Clusters: Enforcement vs Fatality Rate', fontweight='bold')
ax.legend()
fig.tight_layout()
fig

### How to read the cluster scatter:

Each dot is a state, colored by its assigned cluster. States that look similar to the algorithm are the same color.

- **Top-left corner** = high fatality rate, low arrest rate (worst outcome: people dying and nobody being arrested)
- **Bottom-right** = low fatality rate, high arrest rate (aggressive enforcement correlating with fewer deaths)
- **Top-right** = high on both (lots of arrests AND lots of deaths — enforcement isn't working?)

**Look for:** Clusters that overlap on one axis but separate on another. That tells you which variable actually distinguishes the groups.

---
## 5. Consumption-Fatality Gap (Residual Analysis)

Which states have unexpectedly high or low fatality rates given their consumption level?

- **Positive residuals** = more deaths than consumption alone predicts (enforcement/infrastructure problem)
- **Negative residuals** = fewer deaths than expected (effective policy despite drinking)

In [ ]:
# Simple regression: fatality rate ~ consumption
gap_model = smf.ols('alcohol_fatality_rate_per_100m_vmt ~ ethanol_per_capita_gallons_2022', data=df).fit()
df['_residual'] = gap_model.resid

print(f'R² = {gap_model.rsquared:.3f} — consumption explains {gap_model.rsquared*100:.1f}% of fatality rate variance')
print(f'\nStates with HIGHER fatality rates than consumption predicts (top 10):')
print(df.nlargest(10, '_residual')[['state_name', 'ethanol_per_capita_gallons_2022',
      'alcohol_fatality_rate_per_100m_vmt', '_residual']].to_string(index=False))
print(f'\nStates with LOWER fatality rates than consumption predicts (bottom 10):')
print(df.nsmallest(10, '_residual')[['state_name', 'ethanol_per_capita_gallons_2022',
      'alcohol_fatality_rate_per_100m_vmt', '_residual']].to_string(index=False))

### How to read residuals:

- **R²** = how much of the fatality rate is explained by consumption alone. If R² = 0.20, consumption explains 20% — the other 80% is something else (policy, roads, enforcement, culture).
- **Positive residual** = state has MORE deaths than you'd expect based on how much its residents drink. Something else is going wrong (bad roads, lax enforcement, high speeds).
- **Negative residual** = state has FEWER deaths than expected given its consumption. Something is going RIGHT (good policy, enforcement, urban roads).

**Social media angle:** "These states drink just as much but have far fewer deaths — what are they doing differently?" or "These states drink LESS than average but still have the worst fatality rates."

**This is the perception-reality gap** that connects to your body cam hypothesis: states that appear on YouTube as DUI hotspots (Florida, Wisconsin) may actually be outliers in different directions on this chart.

In [ ]:
# Residual plot — annotated scatter
fig, ax = plt.subplots(figsize=(11, 7))

# Regression line
x_range = np.linspace(df['ethanol_per_capita_gallons_2022'].min(), df['ethanol_per_capita_gallons_2022'].max(), 100)
y_pred = gap_model.predict(pd.DataFrame({'ethanol_per_capita_gallons_2022': x_range}))
ax.plot(x_range, y_pred, color='#6B7280', linewidth=1.5, linestyle='--', label='Expected (regression line)')

# Color by residual direction
colors = ['#E76F51' if r > 0 else '#2A9D8F' for r in df['_residual']]
ax.scatter(df['ethanol_per_capita_gallons_2022'], df['alcohol_fatality_rate_per_100m_vmt'],
          c=colors, s=50, alpha=0.8, edgecolors='white', linewidths=0.5)

# Label outliers (top/bottom 5 residuals)
outliers = pd.concat([df.nlargest(5, '_residual'), df.nsmallest(5, '_residual')])
for _, row in outliers.iterrows():
    ax.annotate(row['state_abbr'],
               (row['ethanol_per_capita_gallons_2022'], row['alcohol_fatality_rate_per_100m_vmt']),
               fontsize=8, fontweight='bold', ha='left', va='bottom',
               xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Per Capita Ethanol Consumption (gallons, 2022)')
ax.set_ylabel('Alcohol Fatality Rate per 100M VMT')
ax.set_title('Consumption-Fatality Gap\nRed = worse than expected | Teal = better than expected', fontweight='bold')
ax.legend()
fig.tight_layout()
fig

### How to read this scatter:

- **Dashed line** = what the model predicts: if a state drinks X gallons, this is the fatality rate you'd "expect."
- **Coral dots (above the line)** = worse than expected — more deaths than their consumption level predicts
- **Teal dots (below the line)** = better than expected — fewer deaths despite similar or higher consumption
- **Labeled states** = the biggest outliers in each direction

**The story:** States far above the line have a structural problem beyond drinking culture. States far below have figured out how to keep people alive despite high consumption. The labeled outliers are your most interesting social media targets.

In [ ]:
# What explains the gap? Compare high-residual vs low-residual states
df['gap_group'] = pd.cut(df['_residual'], bins=[-np.inf, df['_residual'].quantile(0.33),
                         df['_residual'].quantile(0.67), np.inf],
                         labels=['Better than expected', 'As expected', 'Worse than expected'])

compare_cols = ['max_speed_limit_mph', 'iid_all_offender', 'checkpoints_permitted',
                'dui_arrest_rate_per_100k_reporting', 'pct_impaired_with_prior_dwi']

print('=== What differs between over/under-performing states? ===\n')
gap_comparison = df.groupby('gap_group', observed=True)[compare_cols].mean().round(2)
print(gap_comparison.to_string())

### How to read the gap comparison:

This table splits states into three groups based on their residual (better/as expected/worse) and shows the average policy/enforcement metrics for each group.

**Look for patterns:**
- Do "better than expected" states have lower speed limits? More checkpoints? Higher IID adoption?
- Do "worse than expected" states have higher speeds? Lower arrest rates? More repeat offenders?

**Conclusion:** Whatever differs most between the best and worst groups is your strongest candidate for "what actually makes the difference" — beyond just how much people drink.

**This directly feeds content:** If worse-than-expected states all have high speed limits and no checkpoints, that's a clear policy story for social media.

---
## Summary & Next Steps

Key findings from this analysis to inform social media content:

- [ ] Regional differences significant? Which regions?
- [ ] Strongest predictor of fatality rates?
- [ ] Do IID/felony/checkpoints matter after controlling for confounders?
- [ ] State archetypes — what's the narrative for each cluster?
- [ ] Which states are "perception gaps" — drink less but die more, or drink more but stay safe?

Feed findings back into `04b-viz-social.ipynb` for polished social charts.